# 📦 LDA & QDA
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Linear and Quadratic Discriminant Analysis use probability theory to classify. Instead of fitting a boundary directly (like logistic regression), they model the distribution of X within each class and use Bayes' theorem to classify.

### What you'll learn
- Bayes' theorem as a classification framework
- LDA — linear decision boundaries (assumes shared covariance)
- QDA — quadratic boundaries (each class has its own covariance)
- When LDA/QDA outperform logistic regression
- Comparing all three: LR vs LDA vs QDA

### Dataset: Stock market direction prediction (Smarket)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Smarket

In [ ]:
import pandas as pd
smarket = pd.read_csv(f'{DATA_DIR}/Smarket.csv')
smarket['Direction_num'] = (smarket['Direction'] == 'Up').astype(int)
print(f"Smarket: {smarket.shape}  |  Up days: {smarket['Direction_num'].mean():.1%}")
smarket.head()

## 📐 Part 1 — Bayes' Theorem for Classification

LDA and QDA both use Bayes' theorem:

```
P(Y=k | X=x) = P(X=x | Y=k) × P(Y=k) / P(X=x)
             =   likelihood  ×   prior   / evidence
```

The class with the highest posterior probability wins.

**LDA** assumes Xₖ ~ N(μₖ, Σ) — each class has its own mean but **shared covariance** Σ. This gives *linear* decision boundaries.

**QDA** assumes Xₖ ~ N(μₖ, Σₖ) — each class has its own covariance Σₖ. This gives *quadratic* boundaries. More flexible but needs more data to estimate.

In [ ]:
# Visualize LDA vs QDA decision boundaries
from sklearn.datasets import make_classification
np.random.seed(42)

# Create two scenarios
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scenario 1: Equal covariance → LDA is optimal
X1 = np.vstack([np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 150),
                np.random.multivariate_normal([2,2], [[1,0.5],[0.5,1]], 150)])
y1 = np.array([0]*150+[1]*150)

# Scenario 2: Unequal covariance → QDA wins
X2 = np.vstack([np.random.multivariate_normal([0,0], [[0.5,0],[0,2]], 150),
                np.random.multivariate_normal([2,2], [[2,0],[0,0.5]], 150)])
y2 = np.array([0]*150+[1]*150)

for ax, X, y, title in zip(axes, [X1,X2], [y1,y2],
                            ['Equal covariances → LDA wins', 'Unequal covariances → QDA wins']):
    lda = LinearDiscriminantAnalysis().fit(X, y)
    qda = QuadraticDiscriminantAnalysis().fit(X, y)
    
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                          np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    Z_lda = lda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_lda, colors=['#1e5fa8'], linewidths=2, linestyles='-')
    
    Z_qda = qda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_qda, colors=['#e85d20'], linewidths=2, linestyles='--')
    
    ax.scatter(X[y==0,0], X[y==0,1], color='#1e5fa8', alpha=0.4, s=15)
    ax.scatter(X[y==1,0], X[y==1,1], color='#e85d20', alpha=0.4, s=15)
    
    from matplotlib.lines import Line2D
    ax.legend([Line2D([0],[0],color='#1e5fa8',lw=2),
               Line2D([0],[0],color='#e85d20',lw=2,ls='--')],
              [f'LDA (acc={lda.score(X,y):.2f})',
               f'QDA (acc={qda.score(X,y):.2f})'], fontsize=9)
    ax.set_title(title)

plt.suptitle('LDA vs QDA Decision Boundaries', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 LDA boundary is always a straight line. QDA can curve to fit unequal covariances.")

In [ ]:
# Compare LR, LDA, QDA on Smarket
predictors = ['Lag1','Lag2','Lag3','Lag4','Lag5','Volume']
X = smarket[predictors].values
y = smarket['Direction_num'].values

# Train on 2001-2004, test on 2005 (temporal split — important for market data!)
mask_train = smarket['Year'] < 2005
X_tr, X_te = X[mask_train], X[~mask_train]
y_tr, y_te = y[mask_train], y[~mask_train]

print(f"Train: {X_tr.shape[0]} days (2001-2004)")
print(f"Test: {X_te.shape[0]} days (2005)")

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
    'QDA':                 QuadraticDiscriminantAnalysis(),
}

print("\n=== Accuracy on 2005 Test Data ===")
for name, model in models.items():
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"  {name:<25} {acc:.4f}")

print("\n📌 All three methods are comparable here — the stock market is hard to predict!")
print("   Baseline (always predict 'Up'): {:.4f}".format(y_te.mean()))

In [ ]:
# LDA's discriminant scores — visualize the separation
lda = LinearDiscriminantAnalysis()
lda.fit(X_tr, y_tr)

scores_tr = lda.transform(X_tr)
scores_te = lda.transform(X_te)

fig, ax = plt.subplots(figsize=(9, 4))
for split, scores, y_split, label, alpha in [
    (None, scores_tr, y_tr, 'Train', 0.4),
    (None, scores_te, y_te, 'Test', 0.9)]:
    ax.hist(scores[y_split==0, 0], bins=25, alpha=alpha*0.7,
            color='#1e5fa8', density=True,
            label=f'Down ({label})' if alpha > 0.5 else None)
    ax.hist(scores[y_split==1, 0], bins=25, alpha=alpha*0.7,
            color='#e85d20', density=True,
            label=f'Up ({label})' if alpha > 0.5 else None)

ax.axvline(0, color='black', lw=1.5, ls='--', label='Decision boundary')
ax.set_xlabel('LDA Discriminant Score')
ax.set_ylabel('Density')
ax.set_title('LDA Discriminant Scores — Up vs Down Days\n(overlap = hard classification problem)')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Large overlap between Up/Down scores confirms stock market is hard to predict from lagged returns")

In [ ]:
answers = {
    "q1": "",  # What distributional assumption does LDA make about the features?
    "q2": "",  # What is the key difference between LDA and QDA?
    "q3": "",  # When does LDA outperform logistic regression?
    "q4": "",  # Why do we need a prior P(Y=k) in Bayes' theorem?
    "q5": "",  # If your data has 5 classes and 2 predictors, how many discriminant functions does LDA produce?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "LDA & QDA"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
